## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style("whitegrid")

In [2]:
import os, sys, json
from pathlib import Path
from IPython.display import display

In [64]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
# import ...

In [4]:
def find_project_root(start=None, markers=("pyproject.toml", ".git", "requirements.txt", ".gitignore")):
    p = Path(start or Path.cwd()).resolve()
    for cur in [p, *p.parents]:
        if any((cur / m).exists() for m in markers):
            return cur
    return p
BASE_CODE_DIR_PATH = find_project_root()
DATASET_DIR = BASE_CODE_DIR_PATH / 'datasets'
DATASET_DIR, BASE_CODE_DIR_PATH

(PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/Reference and Learning Content/Deep-Learning/datasets'),
 PosixPath('/Users/vedaangchopra/all_data/complete_technical_work/Reference and Learning Content/Deep-Learning'))

## Data

In [5]:
# TODO: load winequality-red.csv from UCI (use pandas)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
PATH_TO_CSV = DATASET_DIR / 'winequality.csv'

In [11]:
### Creating Custom Wine Dataset Loader
class WineDataLoader(Dataset):
    target_column ='quality'
    def __init__(self,path_to_csv:Path):
        self.csv_data = pd.read_csv(path_to_csv, header=0, delimiter=';')
        self.x = np.array(self.csv_data.drop(columns=self.target_column))
        self.y = np.array(self.csv_data[[self.target_column]])
        self.columns = list(self.csv_data.drop(columns=self.target_column).columns)
        
    
    def __len__(self):
        ''' Return the length of the CSV File'''
        return len(self.csv_data)
    
    def __str__(self) ->str:
        ''' Display head from CSV'''
        display(self.csv_data.head())
        return "Showing top samples"
    
    def __repr__(self):
        return str(self.csv_data.describe())
    
    def __getitem__(self, index):
        return self.x[index], self.y[index]
        
    
winedata= WineDataLoader(PATH_TO_CSV)
print(len(winedata))

1599


In [8]:
print(winedata)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


Showing top samples


In [12]:
df = winedata.csv_data

assert df.shape == (1599, 12)
assert "quality" in df.columns

In [26]:
# TODO: remap quality to 3 classes using boolean masking — low(3-4)->0, mid(5-6)->1, high(7-8)->2
X = winedata.x
y= winedata.y


In [31]:
y = y.copy()

y_new = np.select(
    [
        (y == 3) | (y == 4),
        (y == 5) | (y == 6),
        (y == 7) | (y == 8)
    ],
    [0, 1, 2]
)
assert set(np.unique(y_new).tolist()) == {0, 1, 2}
assert len(y) == 1599

In [35]:
val, counts =np.unique(y_new, return_counts=True)


array([  63, 1319,  217])

In [55]:
(counts/counts.sum()), (counts/counts.sum()*len(val))

(array([0.03939962, 0.82489056, 0.13570982]),
 array([0.11819887, 2.47467167, 0.40712946]))

In [43]:
# TODO: compute class weights as inverse frequency, normalize so they sum to num_classes
class_weights = (counts / counts.sum() * len(val))
print(class_weights)

assert class_weights.shape == (3,)
assert abs(class_weights.sum().item() - 3.0) < 0.01

[0.11819887 2.47467167 0.40712946]


In [59]:
# TODO: features = all columns except quality, normalize, split 80/20, to tensors, DataLoader batch=64
X = winedata.x
X_train, X_test, y_train, y_test = train_test_split(X, y_new, random_state=0, test_size =0.2)

In [68]:
train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

train_loader = DataLoader( train_dataset, shuffle=True,batch_size=64 )
test_loader = DataLoader(test_dataset, shuffle=True,batch_size=64 )

batch_X, batch_y = next(iter(train_loader))
assert batch_X.shape[1] == 11
assert batch_y.dtype == torch.long

## Model

In [ ]:
# TODO: MLP with BN + Dropout — 11->128->BN->ReLU->Drop(0.3)->64->BN->ReLU->Drop(0.3)->3
class WineNet(nn.Module):
    def __init__(self):
        ...
    def forward(self, x):
        ...

model = WineNet()
out = model(batch_X)
assert out.shape == (64, 3)

In [ ]:
# TODO: verify train vs eval output differs due to dropout
model.train()
out_train = model(batch_X)
model.eval()
out_eval = model(batch_X)

assert not torch.allclose(out_train, out_eval)

## BatchNorm

In [ ]:
# TODO: get reference to first BatchNorm layer
bn = ...

# TODO: record running_mean, do one forward in train mode, verify running_mean changed
model.train()
mean_before = bn.running_mean.clone()
_ = model(batch_X)
assert not torch.allclose(mean_before, bn.running_mean)

In [ ]:
# TODO: record running_mean, do one forward in eval mode, verify running_mean is frozen
model.eval()
mean_before = bn.running_mean.clone()
_ = model(batch_X)
assert torch.allclose(mean_before, bn.running_mean)

## LayerNorm

In [ ]:
# TODO: implement LayerNorm from scratch — normalize over last dim, learnable gamma/beta
class MyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        ...
    def forward(self, x):
        ...

x_test = torch.randn(8, 64)
my_ln = MyLayerNorm(64)
pt_ln = nn.LayerNorm(64)

# TODO: copy pt_ln weights into my_ln
...

assert torch.allclose(my_ln(x_test), pt_ln(x_test), atol=1e-5)

## Loss

In [ ]:
# TODO: implement manual cross-entropy from logits — logsumexp trick for numerical stability
def manual_cross_entropy(logits, targets):
    ...

model.eval()
logits = model(batch_X)
loss_manual = manual_cross_entropy(logits, batch_y)
loss_torch = nn.CrossEntropyLoss()(logits, batch_y)
assert torch.allclose(loss_manual, loss_torch, atol=1e-5)

In [ ]:
# TODO: show the double-softmax bug — apply softmax then pass to CrossEntropyLoss
logits = model(batch_X)
wrong_loss = nn.CrossEntropyLoss()(torch.softmax(logits, dim=1), batch_y)
right_loss = nn.CrossEntropyLoss()(logits, batch_y)

# note: wrong_loss will be lower than right_loss because softmax compresses the range
assert wrong_loss < right_loss

## Schedulers

In [ ]:
# TODO: CosineAnnealingLR — record lr for 50 epochs, assert it decreases then increases
model_sched = WineNet()
opt = torch.optim.Adam(model_sched.parameters(), lr=0.01)
scheduler = ...
lrs = []

for _ in range(50):
    # TODO: record lr, step scheduler
    ...

assert lrs[0] > lrs[25]
assert lrs[25] < lrs[49]

In [ ]:
# TODO: StepLR — step_size=10, gamma=0.5, record lr for 30 epochs
opt2 = torch.optim.Adam(model_sched.parameters(), lr=0.01)
scheduler2 = ...
lrs2 = []

for _ in range(30):
    ...

assert abs(lrs2[0] - 0.01) < 1e-6
assert abs(lrs2[10] - 0.005) < 1e-6
assert abs(lrs2[20] - 0.0025) < 1e-6

In [ ]:
# TODO: ReduceLROnPlateau — simulate flat val_loss for 15 steps, verify lr drops
opt3 = torch.optim.Adam(model_sched.parameters(), lr=0.01)
scheduler3 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt3, patience=5, factor=0.1)

for i in range(15):
    # TODO: step with constant loss
    ...

assert opt3.param_groups[0]["lr"] < 0.01

## Gradient clipping

In [ ]:
# TODO: compute gradient norm before and after clipping
model_clip = WineNet()
opt_clip = torch.optim.SGD(model_clip.parameters(), lr=0.1)
loss = nn.CrossEntropyLoss()(model_clip(batch_X), batch_y)
loss.backward()

# TODO: compute total norm before clip
norm_before = ...

# TODO: clip_grad_norm_ with max_norm=1.0
...

# TODO: compute total norm after clip
norm_after = ...

print(f"norm before: {norm_before:.4f}, after: {norm_after:.4f}")
assert norm_after <= 1.0 + 1e-6

In [ ]:
# TODO: demonstrate exploding gradients — multiply loss by 1e6, show norm without clipping
model_explode = WineNet()
loss_big = 1e6 * nn.CrossEntropyLoss()(model_explode(batch_X), batch_y)
loss_big.backward()

norm_exploded = ...
print(f"exploded norm: {norm_exploded:.2f}")
assert norm_exploded > 1000

## Evaluation

In [ ]:
# TODO: train the model for 50 epochs with Adam, CrossEntropyLoss(weight=class_weights)
model = WineNet()
optimizer = ...
criterion = ...

for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        ...
    # TODO: print loss every 10 epochs
    ...

In [ ]:
# TODO: per-class accuracy — no sklearn, use masks
def per_class_accuracy(preds, targets, num_classes):
    # TODO: return tensor of shape (num_classes,)
    ...

model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        # TODO: accumulate predictions and targets
        ...

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)
acc_per_class = per_class_accuracy(all_preds, all_targets, 3)
assert acc_per_class.shape == (3,)
assert (acc_per_class >= 0).all() and (acc_per_class <= 1).all()

In [ ]:
# TODO: build confusion matrix from scratch — no sklearn
def confusion_matrix(preds, targets, num_classes):
    # TODO: return tensor of shape (num_classes, num_classes), rows=true, cols=pred
    ...

cm = confusion_matrix(all_preds, all_targets, 3)
assert cm.shape == (3, 3)
assert cm.sum() == len(all_targets)